### Configuration

In [6]:
import sys, os
import polars as pl
pl.Config.set_tbl_rows(700)
pl.Config.set_tbl_cols(700)

sys.path.append("../..")

### debug code

In [32]:
import os
import json
from collections import Counter
from datetime import datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

runs_root = Path("../../runs")
run_dirs = sorted(p for p in runs_root.iterdir() if p.is_dir())

oof_cols = []
test_cols = []
oof_arrays = []
test_arrays = []

for rd in run_dirs:
    oof_path = rd / "oof.npy"
    test_path = rd / "test.npy"
    if not (oof_path.exists() and test_path.exists()):
        continue

    name = rd.name  # 例: xgb-031-trl6-5fold-s42
    oof = np.load(oof_path)[:750_000]
    test = np.load(test_path)[:250_000]

    # カラム名を run 名にして格納
    oof_arrays.append(oof)
    test_arrays.append(test)
    oof_cols.append(name)
    test_cols.append(name)

# === 横結合して DataFrame 化 ===
oof_df = pl.DataFrame({c: a for c, a in zip(oof_cols, oof_arrays)})
test_df = pl.DataFrame({c: a for c, a in zip(test_cols, test_arrays)})


# === row_id を追加 ===
oof_df = oof_df.with_row_index("row_id")

# === targetを追加 ===
lf = pl.scan_parquet("../../artifacts/features/033/tr_df.parquet")
y = lf.select("target").collect()
oof_df = oof_df.with_columns(y)

### input cell

In [37]:
oof_df.describe()

statistic,row_id,hc-034,lasso-034-trl5-5fold-s42,lgbm-042-trl1-5fold-s42,mlp-033-trl2-5fold-s42,mlp-039-trl18-5fold-s42,mlp-041-trl7-5fold-s42,xgb-023-trl32-s42,xgb-023-trl36-s42,xgb-023-trl44-s42,xgb-023-trl51-s42,xgb-023-trl57-s42,xgb-023-trl58-s42,xgb-026-trl16-s42,xgb-026-trl17-s42,xgb-026-trl18-s42,xgb-026-trl19-s42,xgb-026-trl21-s42,xgb-027-trl14-s42,xgb-027-trl15-s42,xgb-027-trl16-s42,xgb-027-trl17-s42,xgb-027-trl22-s42,xgb-030-trl19-s42,xgb-030-trl20-s42,xgb-031-trl6-10fold-s42,xgb-031-trl6-15fold-s42,xgb-031-trl6-20fold-s42,xgb-031-trl6-5fold-s42,xgb-031-trl6-7fold-s42,xgb-032-trl19-10fold-s42,xgb-033-trl17-5fold-s42,xgb-033-trl7-10fold-s42,xgb-035-trl20-5fold-s42,xgb-035-trl5-5fold-s42,xgb-036-trl4-5fold-s42,xgb-037-trl7-5fold-s42,xgb-038-trl5-5fold-s42,xgb-039-trl16-5fold-s42,xgb-040-trl17-5fold-s42,xgb-042-trl5-5fold-s42,xgb-042-trl6-5fold-s42,target
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0,750000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",374999.5,0.118705,0.120651,0.120218,0.124725,0.121698,0.123855,0.119813,0.11994,0.119317,0.11864,0.119698,0.119493,0.117967,0.118667,0.119412,0.119389,0.117464,0.119486,0.119463,0.119356,0.119172,0.119315,0.117972,0.117964,0.120072,0.120053,0.120056,0.118961,0.120019,0.120071,0.120454,0.120137,0.120329,0.12022,0.120174,0.119489,0.119541,0.119997,0.114866,0.119243,0.119338,0.120651
"""std""",216506.495284,0.25917,0.218167,0.258408,0.26242,0.253103,0.260323,0.260081,0.260524,0.261207,0.26286,0.260858,0.261718,0.25869,0.258619,0.257462,0.258101,0.260672,0.259689,0.260148,0.260768,0.262235,0.261377,0.257082,0.257041,0.261458,0.261582,0.261406,0.259723,0.261061,0.261109,0.260607,0.260501,0.247655,0.250428,0.248108,0.25808,0.258222,0.258121,0.274096,0.258812,0.26041,0.325721
"""min""",0.0,0.000004,0.020586,1.6066e-7,1.5286e-9,0.0,0.0,0.000002,0.000005,0.000003,0.000001,0.000002,0.000001,0.000002,6.8953e-7,0.000001,1.9221e-7,0.000002,0.000003,0.000003,0.000003,0.000001,0.000002,0.000005,0.000004,4.3083e-7,2.5160e-7,2.4459e-7,4.6109e-7,4.6022e-7,0.000001,0.000001,0.000007,0.000001,0.000011,0.000004,0.000003,0.000003,0.000006,2.4242e-7,0.000008,0.000001,0.0
"""25%""",187500.0,0.000141,0.020873,0.000061,0.000057,0.000076,0.000021,0.000152,0.000163,0.000133,0.000105,0.000126,0.000109,0.000126,0.000123,0.000142,0.000105,0.000137,0.000145,0.000146,0.000137,0.000119,0.000134,0.000171,0.00017,0.000087,0.000085,0.000087,0.000093,0.000093,0.000128,0.000136,0.000168,0.000259,0.000385,0.000306,0.000226,0.000247,0.000266,0.000029,0.000164,0.000113,0.0
"""50%""",375000.0,0.000853,0.021452,0.000657,0.000691,0.000848,0.000413,0.000932,0.000948,0.000849,0.000694,0.000842,0.00076,0.000763,0.000808,0.000886,0.000808,0.000762,0.000859,0.000835,0.000797,0.000713,0.000771,0.000946,0.000952,0.000718,0.000703,0.000717,0.000728,0.00074,0.000851,0.000891,0.000932,0.001676,0.001805,0.001738,0.001199,0.001284,0.001331,0.000192,0.000899,0.000743,0.0
"""75%""",562499.0,0.045864,0.05934,0.051781,0.060497,0.072599,0.063691,0.048213,0.047706,0.044886,0.039629,0.046743,0.044567,0.044144,0.046576,0.050306,0.049361,0.039623,0.047238,0.046305,0.044977,0.042083,0.043974,0.046669,0.046797,0.046514,0.046216,0.046548,0.045983,0.047062,0.047322,0.049478,0.048463,0.073617,0.066864,0.071885,0.050555,0.049419,0.051654,0.015637,0.048213,0.046506,0.0
"""max""",7

In [30]:
coef_list = result["fi_mean"]
coef_arr = np.stack(coef_list, axis=0)
abs_coef_mean = np.mean(np.abs(coef_arr), axis=0).ravel()
print(len(abs_coef_mean))

19


In [31]:
trainer.features

['hc-034',
 'xgb-023-trl32-s42',
 'xgb-023-trl36-s42',
 'xgb-023-trl44-s42',
 'xgb-023-trl51-s42',
 'xgb-023-trl57-s42',
 'xgb-023-trl58-s42',
 'xgb-026-trl16-s42',
 'xgb-026-trl17-s42',
 'xgb-026-trl18-s42',
 'xgb-026-trl19-s42',
 'xgb-026-trl21-s42',
 'xgb-027-trl14-s42',
 'xgb-027-trl15-s42',
 'xgb-027-trl16-s42',
 'xgb-027-trl17-s42',
 'xgb-027-trl22-s42',
 'xgb-030-trl19-s42',
 'xgb-030-trl20-s42']

In [28]:
pl.DataFrame({"x": abs_coef_mean.get()})

x
f32
0.218152
0.0
0.0
0.0
0.0
0.000012
0.0
0.0
0.0


In [40]:
import kaggle

In [41]:
help(kaggle)

Help on package kaggle:

NAME
    kaggle - # coding=utf-8

PACKAGE CONTENTS
    api (package)
    cli
    configuration
    models (package)
    test (package)

DATA
    api = <kaggle.api.kaggle_api_extended.KaggleApi object>

FILE
    /home/hanse/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/kaggle/__init__.py




In [43]:
api = kaggle.KaggleApi()
api.authenticate()